# Kolokvijum II — teorija iza rešenja

Ovaj notebook prati **samo rešenje**, deo po deo. Za svaki mehanizam objašnjava zašto je tu,
koju rečenicu iz zadatka rešava, i daje jednu sliku koja ga čini očiglednim.

*(Ako tražiš koncepte odvojeno od zadatka, na generičkim primerima — to je `k2_teorija.ipynb`.
Ovaj notebook je uz kod koji predaješ.)*

## Rečenica iz zadatka → mehanizam

| Šta zadatak traži | Čime je rešeno | Odeljak |
| --- | --- | --- |
| „Definicija predstavlja funkciju koja se izvršava pozivom execute" | funkcija kao polje objekta | 1 |
| „execute može primiti proizvoljan broj argumenata" | rest i spread `(...argumenti)` | 1 |
| „execute metode se izvršavaju asinhrono" | `async` | 1 i 4 |
| „Workflow je iterabilni objekat" | `Symbol.iterator` + `function*` | 2 |
| „definisati operacije map, flat, flatMap i filter" | metode koje vraćaju **nov Workflow** | 3 |
| „kao ulaz dostavlja rezultat prethodno izvršenog" | petlja sa `await`, ulaz se prenosi | 4 |
| „za prvi task uzimaju se svi prosleđeni argumenti" | akumulator je **niz argumenata** | 4 |
| „proizvoljna kompozicija transdjusera" | `preslikaj`, `izdvoj`, `poredjaj`, `primeni` | 5 i 6 |
| „transdjuser za ispis svakog koraka" | dekorator kao `preslikaj` | 7 |

Sve ćelije se izvršavaju redom i grade rešenje kakvo predaješ.

---
# 1 · Task — posao kao podatak

## Slika: daljinski upravljač

Daljinski ima **jedno dugme**. Dugme je uvek isto — pritisneš ga i nešto se desi. Ali *šta* se desi
zavisi od toga na šta je daljinski **programiran**.

- `execute` je dugme. Uvek se isto zove, uvek se isto pritiska.
- `definicija` je programacija. To je ono što se stvarno izvrši.

Dva `Task` objekta su dva daljinska sa istim dugmetom i različitom programacijom. Zato `Workflow`
može da ih niže ne znajući šta koji radi — njemu treba samo dugme.

## Tri sitnice u jednoj liniji

```js
this.execute = async function (...argumenti) {
  return definicija(...argumenti);
};
```

**`...argumenti` u zagradi** skuplja koliko god argumenata stigne u niz. To je odgovor na
„execute može primiti proizvoljan broj argumenata".

**`...argumenti` u pozivu** ih razlaže nazad u pojedinačne argumente. Isti znak, suprotan smer:
u parametrima **pakuje**, u pozivu **raspakuje**.

**`async`** čini da `execute` **uvek** vrati `Promise`, čak i kad je definicija trenutna. Zbog toga
`Workflow` ne mora da pita da li je neki zadatak spor — svi izgledaju isto.

In [ ]:
const Task = function (naziv, definicija) {
  this.naziv = naziv;
  this.definicija = definicija;

  this.execute = async function (...argumenti) {
    return definicija(...argumenti);
  };
};

// dva daljinska, isto dugme, druga programacija
const zbir = new Task("zbir", (niz) => niz.reduce((a, x) => a + x, 0));
const spoji = new Task("spoji", (a, b, c) => a + "-" + b + "-" + c);

console.log("isto dugme:", typeof zbir.execute, typeof spoji.execute);
console.log("druga programacija:", await zbir.execute([1, 2, 3]), "|", await spoji.execute("a", "b", "c"));

// rest pakuje, spread raspakuje
const koliko = new Task("koliko", (...sve) => sve.length);
console.log("\njedan argument:", await koliko.execute(1));
console.log("tri argumenta: ", await koliko.execute(1, 2, 3));

// async: uvek Promise, i kad je definicija trenutna
console.log("\nbez await:", zbir.execute([1, 2, 3]));
console.log("sa await: ", await zbir.execute([1, 2, 3]));

---
# 2 · Iterabilnost — `Symbol.iterator` i `function*`

## Slika: redomat u banci

Aparat na zidu. Pritisneš — izađe jedan papirić. Pritisneš opet — sledeći. Kad se rolna potroši,
aparat kaže da je gotovo.

Da bi nešto bilo „redomat", **ne mora da bude spisak**. Mora samo da ima dugme na koje se pritiska,
i to dugme se u JavaScript-u uvek zove isto: `Symbol.iterator`.

Zbog toga zadatak i kaže „Workflow **je** iterabilni objekat", a ne „Workflow sadrži niz" — traži se
sposobnost, ne tip.

## Šta `function*` štedi

Bez generatora, dugme moraš da napišeš ručno — sa brojačem, `next()` i `done`:

```js
[Symbol.iterator]() {
  let i = 0;
  return { next: () => i < zadaci.length
                         ? { value: zadaci[i++], done: false }
                         : { value: undefined, done: true } };
}
```

Sa `function*` isto to je jedan red. Zvezdica znači: *ova funkcija ne vraća vrednost nego aparat*.
`yield` je „evo jednog papirića, sad stani".

A `yield*` je „ne izdaj mene, nego **sve** iz ovoga" — skraćenica za petlju.

In [ ]:
const zadaciProbni = ["parsiraj", "zbir"];

// tri načina, isti ishod
const rucno = {
  [Symbol.iterator]() {
    let i = 0;
    return { next: () => i < zadaciProbni.length
                           ? { value: zadaciProbni[i++], done: false }
                           : { value: undefined, done: true } };
  },
};
const petljom = { *[Symbol.iterator]() { for (const z of zadaciProbni) yield z; } };
const zvezdicom = { *[Symbol.iterator]() { yield* zadaciProbni; } };

console.log("rucno:    ", [...rucno]);
console.log("petljom:  ", [...petljom]);
console.log("zvezdicom:", [...zvezdicom]);

// yield naspram yield*
function* jedan() { yield ["a", "b"]; }
function* svi() { yield* ["a", "b"]; }
console.log("\nyield  daje:", [...jedan()], "— jedna vrednost, a to je niz");
console.log("yield* daje:", [...svi()], "— dve vrednosti");

// jedno dugme otkljucava cetiri stvari
console.log("\nfor...of, spread, Array.from, razlaganje:");
for (const z of zvezdicom) console.log("   ", z);
const [prvi, drugi] = zvezdicom;
console.log("   ", [...zvezdicom], Array.from(zvezdicom), prvi, drugi);

### Zašto dodeljuješ funkciju, a ne gotov aparat

```js
this[Symbol.iterator] = function* () { ... };   // funkcija
```

Svaki `for...of` **iznova zove** tu funkciju i dobija **svež** aparat. Da si sačuvao već napravljen
generator, prvi prolaz bi ga potrošio i drugi bi vratio prazno. Otud isti `Workflow` može da se
iterira koliko god puta.

---
# 3 · `map`, `filter`, `flat`, `flatMap` — zašto vraćaju Workflow

## Slika: kutija, ne gomila

Zamisli kutiju sa stvarima. Četiri radnje nad njom:

| Radnja | Šta uradi |
| --- | --- |
| `map` | svaku stvar zameni drugom |
| `filter` | neke izbaci |
| `flat` | otvori kutije koje su unutar kutije i prospe ih u glavnu |
| `flatMap` | svaku stvar zameni kutijicom, pa sve kutijice odmah otvori |

Presudno: sve četiri vraćaju **kutiju**, ne gomilu na stolu. Da vraćaju goli niz, lanac bi se
prekinuo posle prvog koraka — `wf.filter(...).map(...)` ne bi radilo jer niz nema `map` koji vraća
`Workflow`.

## Zašto `flat` uopšte ima smisla

Zato što `Workflow` može da sadrži drugi `Workflow`. A pošto je `Workflow` iterabilan (odeljak 2),
izravnavanje ne mora da pita „da li si ti Workflow" — pita **„da li imaš dugme"**:

```js
typeof z[Symbol.iterator] === "function" ? [...z] : [z]
```

Task nema `Symbol.iterator` → ostaje kakav jeste. Workflow ima → raspakuje se.
Isti red bi radio i za niz, i za bilo šta što neko kasnije napravi.

In [ ]:
const izravnaj = function (zadaci) {
  return zadaci.flatMap((z) =>
    typeof z[Symbol.iterator] === "function" ? [...z] : [z],
  );
};

const Workflow = function (naziv, autor, zadaci = []) {
  this.naziv = naziv;
  this.autor = autor;

  Object.defineProperty(this, "zadaci", {
    get() { return [...zadaci]; },
    enumerable: true,
  });

  this[Symbol.iterator] = function* () { yield* zadaci; };

  this.map = function (f) { return new Workflow(naziv, autor, zadaci.map(f)); };
  this.filter = function (p) { return new Workflow(naziv, autor, zadaci.filter(p)); };
  this.flat = function () { return new Workflow(naziv, autor, izravnaj(zadaci)); };
  this.flatMap = function (f) { return new Workflow(naziv, autor, izravnaj(zadaci.map(f))); };

  this.execute = async function (...argumenti) {
    let ulaz = argumenti;
    for (const zadatak of zadaci) {
      ulaz = [await zadatak.execute(...ulaz)];
    }
    return ulaz[0];
  };

  this.transduce = function (transdjuser, kraj = sakupi, pocetno = []) {
    return zadaci.reduce(transdjuser(kraj), pocetno);
  };

  this.primeni = function (...transdjuseri) {
    return new Workflow(naziv, autor, zadaci.reduce(poredjaj(...transdjuseri)(sakupi), []));
  };
};

// ── zadaci za sve dalje primere ──────────────────────────────────
const parsiraj = new Task("parsiraj", (t) => t.split(",").map(Number));
const bezNula = new Task("bezNula", (n) => n.filter((x) => x !== 0));
const formatiraj = new Task("formatiraj", (x) => "ukupno: " + x);
const cekaj = new Task("cekaj", async (x) => {
  await new Promise((k) => setTimeout(k, 20));
  return x;
});

const priprema = new Workflow("priprema", "B", [parsiraj, bezNula]);
const racun = new Workflow("racun", "B", [cekaj, zbir]);
const glavni = new Workflow("glavni", "B", [priprema, racun, formatiraj]);

console.log("glavni sadrzi dva Workflow-a i jedan Task:", [...glavni].map((z) => z.naziv));
console.log("posle flat:", [...glavni.flat()].map((z) => z.naziv));

const ravan = glavni.flat();
console.log("\nfilter:", [...ravan.filter((z) => z.naziv !== "zbir")].map((z) => z.naziv));
console.log("map:   ", [...ravan.map((z) => new Task(z.naziv.toUpperCase(), z.definicija))].map((z) => z.naziv));
console.log("flatMap:", [...priprema.flatMap((z) => new Workflow("p", "B", [z, new Task("kroz", (x) => x)]))].map((z) => z.naziv));

console.log("\nlanac radi jer svaka vraca Workflow:", [...ravan.filter((z) => z.naziv !== "cekaj").map((z) => z)].length);
console.log("original netaknut:", [...glavni].map((z) => z.naziv));

---
# 4 · `execute` — štafeta

## Slika: štafetna trka

Četiri trkača. Prvi kreće sa **palicom koju mu daš na startu**. Svaki sledeći **ne sme da potrči**
dok ne primi palicu od prethodnog. Vreme koje se meri je vreme **poslednjeg** trkača na cilju.

Prevedeno:

| Štafeta | Zadatak |
| --- | --- |
| palica | rezultat prethodnog zadatka |
| prvi trkač dobija palicu sa starta | „za prvi task uzimaju se **svi prosleđeni argumenti**" |
| ne sme da potrči bez palice | `await` — čeka prethodni |
| poslednji donosi rezultat | „povratna vrednost je rezultat **poslednjeg**" |

```js
this.execute = async function (...argumenti) {
  let ulaz = argumenti;                              // palica sa starta = SVI argumenti
  for (const zadatak of zadaci) {
    ulaz = [await zadatak.execute(...ulaz)];         // primi, otrči, predaj dalje
  }
  return ulaz[0];                                    // poslednji na cilju
};
```

## Zašto je `ulaz` **niz**, a ne vrednost

To je jedina suptilnost u celoj metodi. Prvi zadatak sme da dobije **više** argumenata, svaki
sledeći dobija **tačno jedan** (rezultat prethodnog). Da bi oba slučaja prošla kroz isti red koda,
kroz petlju se nosi **niz argumenata**, pa se razlaže sa `...ulaz`.

Na startu je taj niz sve što je stiglo u `execute`. Posle svakog koraka to je jednočlani niz sa
rezultatom. Otud `ulaz[0]` na kraju.

In [ ]:
const saberiTri = new Task("saberiTri", (a, b, c) => a + b + c);
const puta10 = new Task("puta10", (x) => x * 10);

// prvi trkac dobija SVE sa starta
console.log("tri argumenta na startu:", await new Workflow("t", "B", [saberiTri, puta10]).execute(2, 3, 5));
console.log("   (2+3+5) * 10 = 100 — puta10 je dobio samo rezultat");

// palica se predaje redom
console.log("\nlancano:", await ravan.execute("1,0,2,3"));
console.log("   '1,0,2,3' -> [1,0,2,3] -> [1,2,3] -> 6 -> tekst");

// ne sme da potrci bez palice
const trag = [];
const spor = new Task("spor", async (x) => {
  await new Promise((k) => setTimeout(k, 30));
  trag.push("spor");
  return x;
});
const brz = new Task("brz", (x) => { trag.push("brz"); return x; });

await new Workflow("stafeta", "B", [spor, brz]).execute(1);
console.log("\nredosled:", trag, "— brz je cekao spor, iako je brz");

// ugnjezden workflow prolazi kao i task, jer i on ima execute
console.log("\nugnjezden:", await glavni.execute("1,0,2,3"), "| izravnat:", await ravan.execute("1,0,2,3"));

---
# 5 · Redjuser — ko stoji na kraju

## Slika: šta čeka na kraju trake

Kroz pogon ide traka. Na samom kraju stoji neko i odlučuje šta se dešava sa svakom stvari koja
stigne do njega:

- slaže ih u gajbu → dobiješ **niz**
- samo ih broji → dobiješ **broj**
- meri im težinu → dobiješ **zbir**

Traka je ista, stvari su iste. Menja se **ko stoji na kraju**, i time se menja ishod.

To je redjuser: `(dosadašnje stanje, nova stavka) → novo stanje`.

```js
const sakupi = function (niz, element) { return [...niz, element]; };
const prebroj = function (broj) { return broj + 1; };
```

`sakupi` slaže u niz. `prebroj` ni ne gleda stavku — samo uvećava brojač, pa mu drugi parametar
nije ni potreban.

In [ ]:
const sakupi = function (niz, element) { return [...niz, element]; };
const prebroj = function (broj) { return broj + 1; };

const brojevi = [1, 2, 3, 4];
console.log("isti niz, tri kraja trake:");
console.log("   sakupi: ", brojevi.reduce(sakupi, []));
console.log("   prebroj:", brojevi.reduce(prebroj, 0));
console.log("   zbir:   ", brojevi.reduce((a, x) => a + x, 0));

---
# 6 · Transdjuser — radnik pored trake

## Slika: radnici duž trake

Pored trake stoje radnici. Svaki radi **jednu stvar** sa onim što naiđe, pa to **doda sledećem**.
Radnik ne zna ko je posle njega — njemu se to *kaže*. I ne zna šta se dešava na kraju trake.

Zato radnik nije samo posao — on je **posao plus veza ka sledećem**:

```js
const preslikaj = function (posao) {
  return function (sledeci) {                       // kome predajem
    return function (stanje, element) {             // ovo je opet redjuser
      return sledeci(stanje, posao(element));
    };
  };
};
```

Tri nivoa, tri različita trenutka: *šta radim* (jednom), *kome predajem* (jednom pri sastavljanju),
*sam rad* (za svaku stavku).

## Ovo je mesto gde se najviše greši

**Kroz traku prolaze ZADACI, ne podaci koje zadaci obrađuju.**

U K2 postoje dve različite reke i lako se pomešaju:

| | Šta teče | Čime se obrađuje |
| --- | --- | --- |
| **`execute`** | podaci — `"1,0,2,3"`, `[1,2,3]`, `6` | zadaci, jedan za drugim |
| **transdjuser** | **zadaci** — `parsiraj`, `bezNula`, … | radnici pored trake |

Kad napišeš `preslikaj(z => z.naziv)`, to `z` je **Task**, ne broj. Transdjuser radi nad
**spiskom zadataka**, pre nego što se ijedan izvrši.

## `poredjaj` — raspored radnika

Ide unazad kroz spisak, da bi **redosled pisanja bio i redosled na traci**. Prvi napisani radnik
je onaj koji prvi dohvati stavku.

In [ ]:
const preslikaj = function (posao) {
  return function (sledeci) {
    return function (stanje, element) {
      return sledeci(stanje, posao(element));
    };
  };
};

const izdvoj = function (provera) {
  return function (sledeci) {
    return function (stanje, element) {
      return provera(element) ? sledeci(stanje, element) : stanje;
    };
  };
};

const poredjaj = function (...transdjuseri) {
  return function (kraj) {
    let veza = kraj;
    for (let i = transdjuseri.length - 1; i >= 0; i--) veza = transdjuseri[i](veza);
    return veza;
  };
};

// kroz traku prolaze ZADACI
console.log("nazivi zadataka:", ravan.transduce(preslikaj((z) => z.naziv)));
console.log("broj zadataka:  ", ravan.transduce(preslikaj((z) => z), prebroj, 0));

console.log("\nisti radnik, drugi kraj trake — radnik se ne menja");

// kompozicija: izdvoj pa preslikaj, u JEDNOM prolazu
console.log("\nbez cekaj, pa nazivi:",
  ravan.transduce(poredjaj(izdvoj((z) => z.naziv !== "cekaj"), preslikaj((z) => z.naziv))));

// dokaz da je prolaz jedan
const dnevnikDva = [];
ravan.zadaci
  .filter((z) => { dnevnikDva.push("provera " + z.naziv); return z.naziv !== "cekaj"; })
  .map((z) => { dnevnikDva.push("uzimanje " + z.naziv); return z.naziv; });

const dnevnikJedan = [];
ravan.transduce(poredjaj(
  izdvoj((z) => { dnevnikJedan.push("provera " + z.naziv); return z.naziv !== "cekaj"; }),
  preslikaj((z) => { dnevnikJedan.push("uzimanje " + z.naziv); return z.naziv; }),
));

console.log("\nfilter pa map:", dnevnikDva.join(" -> "));
console.log("transdjuseri: ", dnevnikJedan.join(" -> "));

### Pročitaj ta dva reda

**`filter` pa `map`** proveri **sve** zadatke, pa tek onda počne uzimanje. Između te dve faze mora
negde da stoji međuniz.

**Transdjuseri** puste svaki zadatak kroz oba radnika pre nego što sledeći krene. Nema međuniza.
I vidi se da posle `provera cekaj` nema uzimanja — izbačen je i nikad nije stigao do drugog radnika.

---
# 7 · `saIspisom` — dekorator kao transdjuser

## Slika: prevodilac na sastanku

Dva čoveka pregovaraju preko prevodioca. Sve što prvi kaže prolazi kroz prevodioca do drugog, i
odgovor se vraća istim putem. Sadržaj se **ne menja** — ali prevodilac sve čuje i sve može da zabeleži.

Ni jedan ni drugi ne moraju da znaju da je tu. Za njih se ništa nije promenilo.

To je dekorisanje: **novi sloj sa istim ugovorom, koji oko originala dopisuje posao.**

## Zašto je to samo `preslikaj`

Traženi transdjuser ne mora nikakvu novu mašineriju. Element trake je `Task`, pa je „obmotaj svaki
zadatak" obično preslikavanje — samo što novi element nastaje kao **nov Task oko starog**:

```js
const saIspisom = preslikaj(function (zadatak) {
  return new Task(zadatak.naziv, async function (...argumenti) {
    console.log("   ->", zadatak.naziv, "ulaz:", ...argumenti);
    const rezultat = await zadatak.execute(...argumenti);   // original radi svoj posao
    console.log("   <-", zadatak.naziv, "izlaz:", rezultat);
    return rezultat;                                       // isti rezultat kao da sloja nema
  });
});
```

Tri osobine koje to čine upotrebljivim: **isti ugovor** (prima i vraća isto), **original netaknut**
(i dalje postoji sam za sebe), i **slaganje** (ispred njega može stati još neko).

In [ ]:
const saIspisom = preslikaj(function (zadatak) {
  return new Task(zadatak.naziv, async function (...argumenti) {
    console.log("   ->", zadatak.naziv, "ulaz:", ...argumenti);
    const rezultat = await zadatak.execute(...argumenti);
    console.log("   <-", zadatak.naziv, "izlaz:", rezultat);
    return rezultat;
  });
});

console.log("bez sloja:", await ravan.execute("1,0,2,3"));

console.log("\nsa slojem — isti rezultat, samo se sve vidi:");
console.log("rezultat:", await ravan.primeni(saIspisom).execute("1,0,2,3"));

console.log("\nproizvoljna kompozicija — izbaci cekaj, pa obmotaj:");
const bezCekanja = ravan.primeni(izdvoj((z) => z.naziv !== "cekaj"), saIspisom);
console.log("rezultat:", await bezCekanja.execute("4,0,5"));

console.log("\noriginal nije ni pipnut:", ravan.zadaci[0] === parsiraj);
console.log("i dalje daje isto:", await ravan.execute("1,0,2,3"));

---
# Kako se sve spaja

Rešenje nije sedam nezavisnih delova nego jedan lanac:

1. **Task** drži posao kao podatak (1) → zato `Workflow` može da ga niže ne znajući šta radi.
2. **`Symbol.iterator`** čini `Workflow` iterabilnim (2) → zato `for…of`, spread i izravnavanje rade sami.
3. Pošto je iterabilan, **`flat`** može da raspakuje ugnježđen `Workflow` ne pitajući ga za tip (3).
4. **`execute`** prenosi palicu kroz zadatke (4) → a pošto i `Workflow` ima `execute`, ugnježđen
   prolazi kao običan zadatak.
5. **Redjuser** kaže šta je ishod (5), **transdjuser** šta se usput radi (6) → i oba rade nad
   **spiskom zadataka**, ne nad podacima.
6. **`saIspisom`** je onda samo `preslikaj` koji svaki zadatak zamenjuje obmotanim (7).

## Tri rečenice za odbranu

> **`execute` nosi niz argumenata, ne vrednost.** Zato prvi zadatak sme da dobije više argumenata,
> a svaki sledeći tačno jedan — kroz isti red koda.

> **Kroz transdjuser prolaze zadaci, ne podaci.** `preslikaj(z => z.naziv)` radi nad `Task` objektima,
> pre nego što se ijedan izvrši.

> **`function*` piše `next()` i `done` umesto tebe.** Iterabilnost je sposobnost, ne tip — pitanje
> nije „šta si ti", nego „umeš li da mi daš sledeće".